# Validating Cloud-Native Geospatial Formats

KyFromAbove data uses **Cloud-Optimized GeoTIFF (COG)** for DEMs and orthoimagery, and **Cloud-Optimized Point Cloud (COPC)** for LiDAR. This notebook shows how to validate these formats using abovepy — both for local files and remote URLs.

In [ ]:
import abovepy
print(f"abovepy {abovepy.__version__}")

## 1. Validate a remote COG

abovepy can validate files directly from their S3 URLs using HTTP range requests — no download needed.

In [ ]:
# Search for a DEM tile
tiles = abovepy.search(county="Franklin", product="dem_phase3", max_items=3)
url = tiles.tiles.iloc[0]["asset_url"]
print(f"Validating: {url}")

In [ ]:
# Run built-in validation
result = abovepy.validate(url)
print(result)
print()

for check in result.checks:
    status = "PASS" if check.passed else "FAIL"
    print(f"  [{status}] {check.name}: {check.message}")

## 2. Understanding COG structure

A valid COG has four key properties:

| Property | Why it matters |
|---|---|
| **Internal tiling** | Enables range-request access to spatial subsets without reading the full file |
| **Overviews** | Pre-computed pyramids for fast rendering at different zoom levels |
| **CRS** | Geospatial reference so the data can be placed on a map |
| **Compression** | Reduces file size and transfer time (LZW, Deflate, or ZSTD) |

A GeoTIFF missing tiling or overviews will still *work* but requires downloading the entire file for any operation — defeating the purpose of cloud-native access.

In [ ]:
# Inspect tiling details
tiling = next(c for c in result.checks if c.name == "internal_tiling")
print(f"Tile size: {tiling.detail}")

overviews = next(c for c in result.checks if c.name == "has_overviews")
print(f"Overview levels: {overviews.detail}")

dims = next(c for c in result.checks if c.name == "dimensions")
print(f"Dimensions: {dims.detail}")

## 3. Batch validation with SearchResult

Spot-check multiple tiles from a search result:

In [ ]:
tiles = abovepy.search(county="Franklin", product="dem_phase3", max_items=10)
print(f"Found {tiles.count} tiles")

# Validate first 3 tiles
results = tiles.validate_format(sample=3)
for r in results:
    print(r)

## 4. Validating orthoimagery

Ortho tiles are also COGs — let's compare their structure to DEMs:

In [ ]:
ortho_tiles = abovepy.search(county="Franklin", product="ortho_phase3", max_items=1)
ortho_url = ortho_tiles.tiles.iloc[0]["asset_url"]

ortho_result = abovepy.validate(ortho_url)
print(ortho_result)
print()

for check in ortho_result.checks:
    status = "PASS" if check.passed else "FAIL"
    print(f"  [{status}] {check.name}: {check.message}")

## 5. Deep validation with rio-cogeo

For thorough validation (IFD ordering, ghost overviews, block alignment), use `deep=True`. This requires [rio-cogeo](https://cogeotiff.github.io/rio-cogeo/):

```bash
pip install rio-cogeo
```

In [ ]:
# Deep validation (requires rio-cogeo)
try:
    deep_result = abovepy.validate(url, deep=True)
    print(deep_result)
    rio_check = next((c for c in deep_result.checks if c.name == "rio_cogeo_validate"), None)
    if rio_check:
        print(f"\nrio-cogeo: {rio_check.message}")
        if rio_check.detail and rio_check.detail.get("warnings"):
            for w in rio_check.detail["warnings"]:
                print(f"  Warning: {w}")
except Exception as e:
    print(f"Deep validation not available: {e}")

## 6. COPC point cloud validation

COPC files use spatial indexing for efficient partial reads. Validation checks the COPC VLR structure, CRS, and point format.

Requires the lidar extra: `pip install abovepy[lidar]`

In [ ]:
# Search for a COPC tile
try:
    laz_tiles = abovepy.search(county="Franklin", product="laz_phase2", max_items=1)
    laz_url = laz_tiles.tiles.iloc[0]["asset_url"]
    print(f"Validating: {laz_url}")

    laz_result = abovepy.validate(laz_url)
    print(laz_result)
    print()
    for check in laz_result.checks:
        status = "PASS" if check.passed else "FAIL"
        print(f"  [{status}] {check.name}: {check.message}")
except Exception as e:
    print(f"COPC validation requires laspy: {e}")

## Summary

| Function | What it validates | Dependencies |
|---|---|---|
| `abovepy.validate(path)` | COG tiling, overviews, CRS, compression | rasterio (core) |
| `abovepy.validate(path, deep=True)` | Full COG spec compliance | rio-cogeo (optional) |
| `abovepy.validate(path.copc.laz)` | COPC VLR, spatial index, point format | laspy (lidar extra) |
| `tiles.validate_format(sample=5)` | Spot-check tiles from search results | rasterio (core) |

For more details, see the [Validating Formats](https://chrislyonsKY.github.io/AbovePy/tutorials/validation/) tutorial in the docs.